# 01. Exploración para la ingesta de datos

**Objetivo:** descargar los CSV desde https://mibici.net/es/datos-abiertos con un rango de fecha establecido, identificar diferentes esquemas y codificación entre los diferentes CSVs.

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from mi_bici import dataset as ds

pd.set_option('display.max_colwidth', None)

2026-10-16 16:03:41.442 | INFO     | mi_bici.config:<module>:11 - PROJ_ROOT path is: /home/chuy/Documents/mna/mlops/TC5061-Proyecto-Equipo11


In [2]:
# Definir rango de fechas

START = '2024-01'
END = '2026-09'

sources = ds.load_sources()
months = ds.select_months(sources, START, END)
print(f'{START} - {END}')
print(f'Meses: {len(months)}')

2024-01 - 2026-09
Meses: 33


In [3]:
# Descargar datos

stations_path = ds.download_file(
    sources['stations'],
    ds.RAW_DIR / 'stations' / sources['stations'].split('/')[-1],
)
paths = ds.download_months(months)

sizes = pd.Series({m: p.stat().st_size / 1e6 for m, p in paths.items()}, name='MB')
print(f'Total: {sizes.sum():,.0f} MB')
sizes.describe()

Total: 1,070 MB


count    33.000000
mean     32.426282
std       3.098069
min      25.448981
25%      30.924814
50%      32.640591
75%      33.908887
max      42.566896
Name: MB, dtype: float64

# Encoding

In [4]:
headers = ds.compare_headers(paths)
headers['encoding'].value_counts()

encoding
utf-8-sig    18
cp1252       15
Name: count, dtype: int64

In [5]:
headers[['month', 'encoding', 'n_columns', 'schema']]

,month,encoding,n_columns,schema
0,2024-01,cp1252,8,0
1,2024-02,cp1252,8,1
2,2024-03,cp1252,8,0
3,2024-04,cp1252,8,0
4,2024-05,cp1252,8,1
5,2024-06,cp1252,8,0
6,2024-07,utf-8-sig,8,0
7,2024-08,cp1252,8,0
8,2024-09,cp1252,8,0
9,2024-10,cp1252,8,0


# Esquemas

In [6]:
# Identificar número de columnas en los archivos
print('Número de columnas por archivo:')
print(headers['n_columns'].value_counts().to_string())

Número de columnas por archivo:
n_columns
8    33


In [7]:
# Revisar la posición de las columnas en los archivos
columns_by_position = pd.DataFrame(headers['columns'].tolist(), index=headers['month'])
columns_by_position.columns = [f'col_{i + 1}' for i in columns_by_position.columns]

mismatch = columns_by_position.ne(pd.Series(ds.EXPECTED_COLUMNS, index=columns_by_position.columns))
print(f'\nArchivos con alguna columna distinta a la esperada: {mismatch.any(axis=1).sum()} de {len(headers)}')
display(columns_by_position[mismatch.any(axis=1)])


Archivos con alguna columna distinta a la esperada: 2 de 33


,col_1,col_2,col_3,col_4,col_5,col_6,col_7,col_8
month,,,,,,,,
2024-02,Viaje_Id,Usuario_Id,Genero,A}äe_nacimiento,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id
2024-05,Viaje_Id,Usuario_Id,Genero,A}äe_nacimiento,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id


In [8]:
schemas = headers.groupby('schema').agg(
    n_months=('month', 'count'),
    months=('month', list),
    columns=('columns', 'first'),
)
schemas

,n_months,months,columns
schema,,,
0,31,"[2024-01, 2024-03, 2024-04, 2024-06, 2024-07, 2024-08, 2024-09, 2024-10, 2024-11, 2024-12, 2025-01, 2025-02, 2025-03, 2025-04, 2025-05, 2025-06, 2025-07, 2025-08, 2025-09, 2025-10, 2025-11, 2025-12, 2026-01, 2026-02, 2026-03, 2026-04, 2026-05, 2026-06, 2026-07, 2026-08, 2026-09]","(Viaje_Id, Usuario_Id, Genero, Año_de_nacimiento, Inicio_del_viaje, Fin_del_viaje, Origen_Id, Destino_Id)"
1,2,"[2024-02, 2024-05]","(Viaje_Id, Usuario_Id, Genero, A}äe_nacimiento, Inicio_del_viaje, Fin_del_viaje, Origen_Id, Destino_Id)"


In [9]:
# Muestreo

for schema, row in schemas.iterrows():
    first_month = row['months'][0]
    sample = ds.read_month(paths[first_month], nrows=5)
    print(f'Esquema {schema} ({first_month})')
    display(sample)
    display(sample.dtypes.to_frame('dtype').T)

Esquema 0 (2024-01)


,Viaje_Id,Usuario_Id,Genero,Año_de_nacimiento,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id
0,31789291,640715,M,1984,2024-01-01 00:05:25,2024-01-01 00:29:37,294,271
1,31789292,2353010,F,1995,2024-01-01 00:07:23,2024-01-01 00:16:37,182,254
2,31789293,1556365,M,1984,2024-01-01 00:07:31,2024-01-01 00:16:37,182,254
3,31789294,2626233,M,1994,2024-01-01 00:07:56,2024-01-01 00:31:51,35,154
4,31789295,2602006,M,1998,2024-01-01 00:08:09,2024-01-01 00:31:51,35,154


,Viaje_Id,Usuario_Id,Genero,Año_de_nacimiento,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id
dtype,int64,int64,object,int64,object,object,int64,int64


Esquema 1 (2024-02)


,Viaje_Id,Usuario_Id,Genero,A}äe_nacimiento,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id
0,32244898,133007,M,1994,2024-02-01 00:01:46,2024-02-01 00:13:00,62,195
1,32244900,2524051,F,1997,2024-02-01 00:02:26,2024-02-01 00:12:41,196,25
2,32244901,1431452,M,2001,2024-02-01 00:02:36,2024-02-01 00:02:39,383,383
3,32244902,1405511,F,1969,2024-02-01 00:02:52,2024-02-01 00:14:38,66,14
4,32244903,276725,M,1990,2024-02-01 00:02:57,2024-02-01 00:12:17,75,179


,Viaje_Id,Usuario_Id,Genero,A}äe_nacimiento,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id
dtype,int64,int64,object,int64,object,object,int64,int64


# Tipo de dato: fecha

In [10]:
# Identificar los diferentes formatos para las columnas de tipo fecha

rows = []
for month, path in paths.items():
    encoding, _ = ds.read_header(path)
    raw = pd.read_csv(path, encoding=encoding, usecols=[4, 5], header=0, names=['start_time', 'end_time'], dtype=str)
    values = pd.concat([raw['start_time'], raw['end_time']]).dropna()
    matched = next(
        (fmt for fmt in ds.DATETIME_FORMATS if pd.to_datetime(values, format=fmt, errors='coerce').notna().all()),
        'NO RECONOCIDO',
    )
    rows.append({'month': month, 'format': matched, 'example': values.iloc[0]})

evidence = pd.DataFrame(rows)
evidence.groupby('format').agg(
    n_months=('month', 'count'),
    months=('month', list),
    example=('example', 'first'),
)

,n_months,months,example
format,,,
%Y-%m-%d %H:%M:%S,31,"[2024-01, 2024-02, 2024-03, 2024-04, 2024-05, 2024-06, 2024-07, 2024-08, 2024-11, 2024-12, 2025-01, 2025-02, 2025-03, 2025-04, 2025-05, 2025-06, 2025-07, 2025-08, 2025-09, 2025-10, 2025-11, 2025-12, 2026-01, 2026-02, 2026-03, 2026-04, 2026-05, 2026-06, 2026-07, 2026-08, 2026-09]",2024-01-01 00:05:25
%d/%m/%Y %H:%M,2,"[2024-09, 2024-10]",01/09/2024 0:00


# Nomenclatura de estaciones

In [11]:
stations = ds.read_month(stations_path)
print(f'Estaciones: {len(stations)}')stations[stations['dpcapacity'] == 0]
display(stations.head())
display(stations.describe(include='all').T)

SyntaxError: invalid syntax (427783987.py, line 2)

In [ ]:
stations[stations['dpcapacity'] == 0]

## Hallazgos y acciones
- El servidor de MiBici responde 406 al User-Agent por defecto de `requests`.
- Se descargaron 33 csv que corresponden a los meses entre Enero del 2024 y Septiembre del 2026. Aprox. 1,070 MB en total. con un promedio de 32 MB por archivo/mes.
- 4 estaciones tienen dpcapacity = 0. Esas estaciones tienen el prefijo "ex-" en el nombre de la estación. Al parecer son esataciones retiradas.
- Los 33 archivos contienen 8 columnas en el mismo orden
- Se identifican 2 esquemas diferentes en los 33 archivos. El esquema 0 lo tienen 31 meses vs esquema 1 con 2 meses. La diferencia del esquema 2 es la columna "Año de nacimiento". El nombre de esta columna en el esquema 2 es "A}äe_nacimiento".
- Se identifican 2 formatos de fecha para las columnas de tipo fecha. **%Y-%m-%d %H:%M:%S** para 31 archivos y **%d/%m/%Y %H:%M** para 2 archivos.
- Se identifican 2 encodings en los archivos: cp1252 en 15 meses y utf-8-sig en 18 meses.
- Los datos incluyen datos personales: Usuario_Id, Genero y Año_de_nacimiento.
- La nomenclatura tiene 484 estaciones; algunas con capacidad 0 (retiradas)